In [11]:
import pandas as pd

df = pd.read_csv('sensor.power_total_load.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.set_index('time')
df = df.asfreq('30min')  # interval is 30 minutes
df.head(30)

,sensor.power_load_no_var_loads,sensor.power_photovoltaics
time,,
2025-06-01 00:00:00+00:00,2279.731128,0.000000
2025-06-01 00:30:00+00:00,2276.004111,0.000000
2025-06-01 01:00:00+00:00,2176.561968,0.000000
2025-06-01 01:30:00+00:00,2246.751068,0.000000
2025-06-01 02:00:00+00:00,2240.592627,0.000000
2025-06-01 02:30:00+00:00,2212.603995,0.000000
2025-06-01 03:00:00+00:00,648.246788,0.000000
2025-06-01 03:30:00+00:00,421.054960,0.000000
2025-06-01 04:00:00+00:00,421.971766,56.621000


In [15]:
#!/usr/bin/env python

import copy
import json
import pathlib
import pickle

import numpy as np
import pandas as pd
from skforecast.recursive import ForecasterRecursive

from emhass import utils
from emhass.command_line import set_input_data_dict
from emhass.machine_learning_forecaster import MLForecaster
from emhass.retrieve_hass import RetrieveHass
from emhass.forecast import Forecast
from emhass.optimization import Optimization
import yaml

# The root folder
root = pathlib.Path.cwd().parent  
# Build emhass_conf paths
emhass_conf = {}
emhass_conf["data_path"] = root / "data/"
emhass_conf["root_path"] = root / "src/emhass/"
emhass_conf["defaults_path"] = emhass_conf["root_path"] / "data/config_defaults.json"
emhass_conf["associations_path"] = emhass_conf["root_path"] / "data/associations.csv"

# create logger
logger, ch = utils.get_logger(__name__, emhass_conf, save_to_file=False)

import yaml
import pathlib
from emhass import utils

# Build config, secrets, and params
config = utils.build_config(emhass_conf, logger, emhass_conf["defaults_path"])
# config["Latitude"] = 1.3521
# config["Longitude"] = 103.8198
# config["time_zone"] = "Asia/Singapore"
_, secrets = utils.build_secrets(emhass_conf, logger, no_response=True)
params = utils.build_params(emhass_conf, secrets, config, logger)

# Parse YAML-like params into config dicts
import json
retrieve_hass_conf, optim_conf, plant_conf = utils.get_yaml_parse(json.dumps(params), logger)

# 3. Forecast load for tomorrow
mlf = MLForecaster(df, "my_model", "sensor.power_load_no_var_loads", "KNeighborsRegressor", 48, emhass_conf, logger)
mlf.fit()
load_forecast = mlf.predict()  # This should give you the next day's load forecast

# 4. Forecast PV for tomorrow
params = json.dumps({"passed_data": {"weather_forecast_cache": False}})
forecast = Forecast(
    retrieve_hass_conf,
    optim_conf,
    plant_conf,
    params=params,  # pass as JSON string
    emhass_conf=emhass_conf,
    logger=logger
)
weather_forecast = forecast.get_weather_forecast(method="open-meteo")  # or "solcast", "csv", etc.
print("Weather forecast head:")
print(weather_forecast.head())
print("Weather forecast shape:", weather_forecast.shape)

pv_forecast = forecast.get_power_from_weather(weather_forecast)
print("PV forecast head:")
print(pv_forecast.head())
print("PV forecast shape:", pv_forecast.shape)


# 5. Prepare DataFrame for optimization
# print("len(load_forecast):", len(load_forecast))
# print("len(load_forecast.index):", len(load_forecast.index))
# print("load_forecast.index:", load_forecast.index)
print("Load forecast head:")
print(load_forecast.head())
print("Load forecast shape:", load_forecast.shape)


df_forecast = pd.DataFrame({
    'sensor.power_load_no_var_loads': load_forecast.values,
    'sensor.power_photovoltaics': pv_forecast.values
}, index=load_forecast.index)
print("df_forecast head:")
print(df_forecast.head())
print("df_forecast shape:", df_forecast.shape)

# 6. Run day-ahead optimization
print("Plant conf:", plant_conf)
print("Optimization conf:", optim_conf)
print("Retrieve HASS conf:", retrieve_hass_conf)
var_load_cost = "sensor.power_load_no_var_loads"
var_prod_price = "sensor.power_photovoltaics"

opt = Optimization(
    retrieve_hass_conf,
    optim_conf,
    plant_conf,
    var_load_cost,
    var_prod_price,
    costfun="profit",
    emhass_conf=emhass_conf,
    logger=logger
)

print("Starting optimization...")
opt_result = opt.perform_dayahead_forecast_optim(df_forecast, pv_forecast, load_forecast)
print("Optimization result head:")
print(opt_result.head())
print("Optimization result columns:", opt_result.columns)

# 1. Baseline cost calculation (no optimization, just using forecasted values and tariffs)
baseline = (-0.001 * opt.timeStep *
            (df_forecast['sensor.power_load_no_var_loads'] * opt_result['unit_load_cost'] +
             df_forecast['sensor.power_photovoltaics'] * opt_result['unit_prod_price'])).sum()
print(f"Baseline cost: {baseline:.2f} €")

# 2. Optimized cost (from the optimization result)
optimized_cost = opt_result["cost_fun_cost"].iloc[-1]
print(f"Optimized cost: {optimized_cost:.2f} €")

# 3. Savings
savings = baseline - optimized_cost
print(f"Savings: {savings:.2f} €")

# 4. Show the optimized schedule (look for columns like 'u_device', 'u_battery', etc.)
print("Optimized schedule columns:")
for col in opt_result.columns:
    if col.startswith('u_') or 'schedule' in col:
        print(col)
print(opt_result[[col for col in opt_result.columns if col.startswith('u_') or 'schedule' in col]].head(10))


2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. config parameters may default to config_defaults.json
2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. config parameters may default to config_defaults.json
2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. config parameters may default to config_defaults.json
2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. config parameters may default to config_defaults.json
2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. config parameters may default to config_defaults.json
2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. config parameters may default to config_defaults.json
2025-07-06 20:15:17,737 - __main__ - INFO - config.json does not exist, or has not been passed. conf

Weather forecast head:
                           temp_air  relative_humidity  precipitable_water  \
2025-07-06 20:30:00+02:00      -5.7                100                 0.1   
2025-07-06 21:00:00+02:00      -5.8                100                 0.1   
2025-07-06 21:30:00+02:00      -5.8                 99                 0.1   
2025-07-06 22:00:00+02:00      -6.0                 98                 0.1   
2025-07-06 22:30:00+02:00      -6.2                 98                 0.1   

                           cloud_cover  wind_speed   ghi   dhi  dni  
2025-07-06 20:30:00+02:00          100        35.7  10.4  10.4  0.0  
2025-07-06 21:00:00+02:00          100        35.2   6.4   6.4  0.0  
2025-07-06 21:30:00+02:00          100        36.1   0.0   0.0  0.0  
2025-07-06 22:00:00+02:00          100        37.5   0.0   0.0  0.0  
2025-07-06 22:30:00+02:00          100        38.9   0.0   0.0  0.0  
Weather forecast shape: (48, 8)


2025-07-06 20:15:19,785 - __main__ - DEBUG - get_power_from_weather returning:
2025-07-06 20:30:00+02:00       3.577229
2025-07-06 21:00:00+02:00       0.000000
2025-07-06 21:30:00+02:00       0.000000
2025-07-06 22:00:00+02:00       0.000000
2025-07-06 22:30:00+02:00       0.000000
2025-07-06 23:00:00+02:00       0.000000
2025-07-06 23:30:00+02:00       0.000000
2025-07-07 00:00:00+02:00       0.000000
2025-07-07 00:30:00+02:00       0.000000
2025-07-07 01:00:00+02:00       0.000000
2025-07-07 01:30:00+02:00       0.000000
2025-07-07 02:00:00+02:00       0.000000
2025-07-07 02:30:00+02:00       0.000000
2025-07-07 03:00:00+02:00       0.000000
2025-07-07 03:30:00+02:00       0.000000
2025-07-07 04:00:00+02:00       0.000000
2025-07-07 04:30:00+02:00       0.000000
2025-07-07 05:00:00+02:00       0.000000
2025-07-07 05:30:00+02:00       0.000000
2025-07-07 06:00:00+02:00       0.000000
2025-07-07 06:30:00+02:00      46.466719
2025-07-07 07:00:00+02:00     175.842049
2025-07-07 07:30:00

PV forecast head:
2025-07-06 20:30:00+02:00    3.577229
2025-07-06 21:00:00+02:00    0.000000
2025-07-06 21:30:00+02:00    0.000000
2025-07-06 22:00:00+02:00    0.000000
2025-07-06 22:30:00+02:00    0.000000
Freq: 30min, dtype: float64
PV forecast shape: (48,)
Load forecast head:
2025-07-03 00:00:00+00:00    2218.572236
2025-07-03 00:30:00+00:00    2264.366437
2025-07-03 01:00:00+00:00    2274.515278
2025-07-03 01:30:00+00:00    2373.563465
2025-07-03 02:00:00+00:00    2300.793686
Freq: 30min, Name: pred, dtype: float64
Load forecast shape: (48,)
df_forecast head:
                           sensor.power_load_no_var_loads  \
2025-07-03 00:00:00+00:00                     2218.572236   
2025-07-03 00:30:00+00:00                     2264.366437   
2025-07-03 01:00:00+00:00                     2274.515278   
2025-07-03 01:30:00+00:00                     2373.563465   
2025-07-03 02:00:00+00:00                     2300.793686   

                           sensor.power_photovoltaics  
2025-0

2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Status: Optimal
2025-07-06 20:15:20,001 - __main__ - INFO - Total value of the Cost function = -10857.29
2025-07-06 20:15:20,001 - __main__ - INFO - Total value of the Cost function = -10857.29
2025-07-06 20:15:20,001 - __main__ - INFO - Total value of the Cost function = -10857.29
2025-07-06 20:15:20,001 - __main__ - INFO - Total value of the Cost funct

Optimization result head:
                               P_PV       P_Load  P_deferrable0  \
2025-07-03 00:00:00+00:00  3.577229  2218.572236            0.0   
2025-07-03 00:30:00+00:00  0.000000  2264.366437            0.0   
2025-07-03 01:00:00+00:00  0.000000  2274.515278            0.0   
2025-07-03 01:30:00+00:00  0.000000  2373.563465            0.0   
2025-07-03 02:00:00+00:00  0.000000  2300.793686            0.0   

                           P_deferrable1  P_grid_pos  P_grid_neg     P_grid  \
2025-07-03 00:00:00+00:00            0.0   2214.9950         0.0  2214.9950   
2025-07-03 00:30:00+00:00            0.0   2264.3664         0.0  2264.3664   
2025-07-03 01:00:00+00:00            0.0   2274.5153         0.0  2274.5153   
2025-07-03 01:30:00+00:00            0.0   2373.5635         0.0  2373.5635   
2025-07-03 02:00:00+00:00            0.0   2300.7937         0.0  2300.7937   

                           unit_load_cost  unit_prod_price  cost_profit  \
2025-07-03 00:00:00+0

KeyError: 'cost_fun_cost'

In [ ]:
#!/usr/bin/env python

import copy
import json
import pathlib
import pickle

import numpy as np
import pandas as pd
from skforecast.recursive import ForecasterRecursive

from emhass import utils
from emhass.command_line import set_input_data_dict
from emhass.machine_learning_forecaster import MLForecaster
from emhass.retrieve_hass import RetrieveHass
from emhass.forecast import Forecast
from emhass.optimization import Optimization
import yaml

import unittest
unittest.main(argv=[''], verbosity=2, exit=False)

# The root folder
root = pathlib.Path.cwd().parent  
# Build emhass_conf paths
emhass_conf = {}
emhass_conf["data_path"] = root / "data/"
emhass_conf["root_path"] = root / "src/emhass/"
emhass_conf["defaults_path"] = emhass_conf["root_path"] / "data/config_defaults.json"
emhass_conf["associations_path"] = emhass_conf["root_path"] / "data/associations.csv"

# create logger
logger, ch = utils.get_logger(__name__, emhass_conf, save_to_file=False)
ch.setLevel("DEBUG")         


class TestMyPipeline(unittest.TestCase):

    @staticmethod
    def get_test_params():
        # --- defaults + secrets ------------------------------------
        cfg  = utils.build_config(emhass_conf, logger,
                                   emhass_conf["defaults_path"])
        cfg.update({"Latitude":1.3521, "Longitude":103.8198,
                    "time_zone":"Asia/Singapore"})          # <- patch here
        _, sec = utils.build_secrets(emhass_conf, logger, no_response=True)
        return utils.build_params(emhass_conf, sec, cfg, logger)

    def setUp(self):
        # ------------------------------------------------------------
        params       = TestMyPipeline.get_test_params()
        runtime      = {
            "historic_days_to_retrieve": 30,
            "model_type": "total_load_1M",
            "var_model": "sensor.power_total_load",
            "sklearn_model": "KNeighborsRegressor",
            "num_lags": 48,
        }
        params["passed_data"] = runtime
        params["optim_conf"]["load_forecast_method"] = "mlforecaster"

        self.params_json        = json.dumps(params)
        self.runtimeparams_json = json.dumps(runtime)

        # input_data_dict now contains df_input_data *and* a Forecast object
        self.input_data_dict = set_input_data_dict(
            emhass_conf, costfun="cost",
            params_json=self.params_json,
            runtimeparams_json=self.runtimeparams_json,
            action="forecast-model-optimization",
            logger=logger, get_data_from_file=True)

        # Short-cuts
        self.df_hist   = copy.deepcopy(self.input_data_dict["df_input_data"])
        self.rh_conf, self.optim_conf, self.plant_conf = utils.get_yaml_parse(
            self.params_json, logger)
        
        
        # -------------- ML forecaster --------------------------------
        r = runtime          # alias
        self.mlf = MLForecaster(self.df_hist, r["model_type"],
                                r["var_model"], r["sklearn_model"],
                                r["num_lags"], emhass_conf, logger)
        self.mlf.fit()                      # keep the fitted object for reuse

        # -------------- Forecast object ------------------------------
        self.fcst = Forecast(self.rh_conf, self.optim_conf, self.plant_conf,
                             self.params_json, emhass_conf, logger,
                             get_data_from_file=True)

    # ----------------------------------------------------------------
    def test_full_pipeline(self):

        # 1. load forecast with fitted mlf
        P_load = self.fcst.get_load_forecast(method="mlforecaster",
                                             mlf=self.mlf,
                                             use_last_window=False,
                                             debug=True)

        # 2. PV forecast (you can switch to solcast, csv, etc.)
        df_weather = self.fcst.get_weather_forecast(method="open-meteo")
        P_PV       = self.fcst.get_power_from_weather(df_weather)

        # 3. dataframe for optimisation
        df_day = pd.concat([P_PV, P_load], axis=1,
                           keys=["P_PV","P_load"])

        # 4. optimisation
        opt = Optimization(self.rh_conf, self.optim_conf, self.plant_conf,
                           var_load_cost=self.fcst.var_load_cost,
                           var_prod_price=self.fcst.var_prod_price,
                           costfun="cost",
                           emhass_conf=emhass_conf, logger=logger)

        df_opt = opt.perform_dayahead_forecast_optim(df_day,
                                                     P_PV.values,
                                                     P_load.values)
                # 5. cost comparison -------------------------------------------------
        baseline = (-0.001 * opt.timeStep *
                    (df_day["P_load"]*df_opt["unit_load_cost"] +
                     df_day["P_PV"]*df_opt["unit_prod_price"])).sum()

        optimised = df_opt["cost_fun_cost"].iloc[-1]  # negative total cost
        savings   = baseline - optimised
        logger.info(f"💰  Saved {savings:.2f} € vs baseline")

        # 6. metrics --------------------------------------------------------
        # last 20 % of historic series as hold-out
        split = int(len(self.df_hist)*0.8)
        y_true = self.df_hist.iloc[split:][self.mlf.var_model]

        y_pred = self.mlf.forecaster.predict(
                     steps=len(y_true),
                     exog=self.df_hist.drop(self.mlf.var_model, axis=1).iloc[split:])

        rmse = np.sqrt(((y_true-y_pred)**2).mean())
        r2   = 1 - ((y_true-y_pred)**2).sum() / ((y_true-y_true.mean())**2).sum()

        # 7. assertions for the unit-test -----------------------------------
        self.assertLess(optimised, baseline,
                        "Optimisation did not save money")
        self.assertGreater(r2, 0.5, "R² too low")
        self.assertLess(rmse, 0.25*y_true.max(),
                        "RMSE still large vs peak load")
        
if __name__ == "__main__":
    unittest.main(verbosity=2)



----------------------------------------------------------------------
Ran 0 tests in 0.000s

NO TESTS RAN
usage: ipykernel_launcher.py [-h] [-v] [-q] [--locals] [--durations N] [-f]
                             [-c] [-b] [-k TESTNAMEPATTERNS]
                             [tests ...]
ipykernel_launcher.py: error: argument -f/--failfast: ignored explicit argument 'c:\\Users\\26293\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3e50773646958a8ab001be46691e1a2299b380668.json'


SystemExit: 2

c:\Users\26293\Desktop\WORK\emhass_git\emhass\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
